# Cakici et al. (2024) — ML and the crypto cross-section (classroom sketch)

**Paper:** Cakici, N., Shahzad, S. J. H., Będowska-Sójka, B., & Zaremba, A. (2024). *Machine learning and the cross-section of cryptocurrency returns.* *International Review of Financial Analysis*, 94, 103244. DOI [10.1016/j.irfa.2024.103244](https://doi.org/10.1016/j.irfa.2024.103244)

This is a **pedagogical sketch**, not a replication. The paper feeds **40 characteristics** on **500+ coins** (2017–2023) to **eight** ML models. We do **not** ship that panel.

A **synthetic weekly coin panel** plants their main claims:

1. a **handful of simple** characteristics (price, past alpha, illiquidity, momentum) carry the signal;
2. the signal is mostly **linear**, so extra **complexity** does not help (the opposite of Gu on equities);
3. the long–short profit lives in the **long** leg and in **hard-to-trade** names.

Magnitudes will **not** match the paper. The design choices should.

**Siblings:** notebook `01a` is the equity Gu ladder; notebook `05` classifies delisting on the same market.

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*encountered in matmul")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(2024)
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

N_COINS = 80
N_WEEKS = 260
TRAIN_WEEKS = 130
VAL_WEEKS = 52
WEEKLY_TO_ANN = np.sqrt(52)
SIGNAL = ["price", "alpha", "illiq", "mom"]
NOISE = ["z0", "z1", "z2", "z3"]
FEATURES = SIGNAL + NOISE

## 1. Synthetic weekly panel

Four **simple** characteristics (paper’s variable-importance headline) plus four noise columns.

True expected return (unknown to the models):

$$
\mathbb{E}[r_{i,t+1}]
= (b_1\,\mathrm{price}+b_2\,\alpha+b_3\,\mathrm{illiq}+b_4\,\mathrm{mom})_{i,t}
\times (1 + c\,\mathbf{1}_{\{\text{hard to trade}\}}).
$$

The mapping is **linear**. Trees and nets have nothing extra to find. The multiplier is larger for small / illiquid / volatile coins — the paper’s limits-to-arbitrage claim.

`mcap` is higher when `illiq` is lower, so value-weighting mutes the names that pay.

In [ ]:
dates = pd.date_range("2018-01-05", periods=N_WEEKS, freq="W-FRI")
coins = np.arange(N_COINS)

illiq = np.zeros((N_WEEKS, N_COINS))
illiq[0] = rng.normal(size=N_COINS)
for t in range(1, N_WEEKS):
    illiq[t] = 0.80 * illiq[t - 1] + 0.55 * rng.normal(size=N_COINS)

price = rng.normal(size=(N_WEEKS, N_COINS))
alpha = rng.normal(size=(N_WEEKS, N_COINS))
mom = rng.normal(size=(N_WEEKS, N_COINS))
vol = 0.40 * illiq + rng.normal(size=(N_WEEKS, N_COINS))
mcap = np.exp(-0.85 * illiq + rng.normal(scale=0.30, size=(N_WEEKS, N_COINS)))
hard = ((illiq > np.quantile(illiq, 0.67)) | (vol > np.quantile(vol, 0.67))).astype(float)

core = 0.004 * price + 0.004 * alpha + 0.005 * illiq + 0.004 * mom
mu = core * (1.0 + 1.2 * hard)
ret = mu + 0.050 * rng.normal(size=(N_WEEKS, N_COINS))

rows = []
for t, dt in enumerate(dates):
    rows.append(pd.DataFrame({
        "date": dt,
        "coin": coins,
        "ret": ret[t],
        "price": price[t],
        "alpha": alpha[t],
        "illiq": illiq[t],
        "mom": mom[t],
        "z0": rng.normal(size=N_COINS),
        "z1": rng.normal(size=N_COINS),
        "z2": rng.normal(size=N_COINS),
        "z3": rng.normal(size=N_COINS),
        "mcap": mcap[t],
        "hard": hard[t],
    }))
panel = pd.concat(rows, ignore_index=True)
print(panel.shape)
print("date range:", panel["date"].min().date(), "→", panel["date"].max().date())
print("share hard-to-trade rows:", round(panel["hard"].mean(), 3))

## 2. Temporal train / validation / test

**Never** shuffle coin–weeks. Tune only on validation; report on test.

In [ ]:
train_end = dates[TRAIN_WEEKS - 1]
val_end = dates[TRAIN_WEEKS + VAL_WEEKS - 1]
train = panel[panel["date"] <= train_end].copy()
val = panel[(panel["date"] > train_end) & (panel["date"] <= val_end)].copy()
test = panel[panel["date"] > val_end].copy()
for name, df in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:5s}: {df['date'].nunique():3d} weeks | {len(df):6d} rows")

## 3. Model ladder (paper spirit: eight families → four classroom stand-ins)

| Classroom | Paper family |
|-----------|--------------|
| OLS | raw regression |
| Ridge | regularized regression |
| RF | trees |
| MLP | neural nets |
| Combo | equal-weight forecast combination (their best VW Sharpe) |

Gu-style $R^2_{\mathrm{OOS}}$ vs a **zero** forecast, then a weekly long–short from the predicted ranking.

In [ ]:
def r2_oos(y, yhat):
    y, yhat = np.asarray(y), np.asarray(yhat)
    ss_res = float(np.sum((y - yhat) ** 2))
    ss_tot = float(np.sum(y ** 2))
    return np.nan if ss_tot <= 0 else 1.0 - ss_res / ss_tot


def fit_models(tr):
    X, y = tr[FEATURES], tr["ret"]
    models = {
        "OLS": LinearRegression(),
        "Ridge": Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=1.0))]),
        "RF": RandomForestRegressor(
            n_estimators=80, max_depth=3, min_samples_leaf=40,
            random_state=2024, n_jobs=-1,
        ),
        "MLP": Pipeline([
            ("sc", StandardScaler()),
            ("m", MLPRegressor(
                hidden_layer_sizes=(8,), max_iter=400,
                random_state=2024, early_stopping=True,
            )),
        ]),
    }
    for m in models.values():
        m.fit(X, y)
    return models


def predict_all(models, df):
    out = {name: m.predict(df[FEATURES]) for name, m in models.items()}
    out["Combo"] = np.mean(list(out.values()), axis=0)
    return out


models = fit_models(train)
pred_val = predict_all(models, val)
pred_test = predict_all(models, test)
rows = []
for name in ["OLS", "Ridge", "RF", "MLP", "Combo"]:
    rows.append({
        "model": name,
        "R2_val": r2_oos(val["ret"], pred_val[name]),
        "R2_test": r2_oos(test["ret"], pred_test[name]),
    })
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("If OLS / Ridge beat RF / MLP on R2, that is the paper’s complexity point — not a bug.")

## 4. Forecasts → weekly long–short

Each week: long the top quintile of $\hat r$, short the bottom. Equal-weighted first.
Pick the model on **validation** Sharpe; then show every model on **test**.

In [ ]:
def long_short(df, yhat, q=0.2, weight=None, cost=0.0):
    tmp = df[["date", "ret", "mcap"]].copy()
    tmp["yhat"] = yhat
    rec = []
    for dt, g in tmp.groupby("date"):
        lo, hi = g["yhat"].quantile([q, 1 - q])
        long, short = g[g["yhat"] >= hi], g[g["yhat"] <= lo]
        if long.empty or short.empty:
            continue
        w = weight

        def wmean(sl):
            if w is None:
                return float(sl["ret"].mean())
            ww = sl[w].to_numpy(float)
            return float(np.average(sl["ret"], weights=ww)) if ww.sum() > 0 else float(sl["ret"].mean())

        rec.append((dt, wmean(long) - wmean(short) - 2 * cost))
    return pd.Series(dict(rec)).sort_index()


def summarise(x, label):
    x = x.dropna()
    mu, sd = float(x.mean()), float(x.std(ddof=1))
    sharpe = (mu / sd) * WEEKLY_TO_ANN if sd > 0 else np.nan
    return {"strategy": label, "mean_w": mu, "sharpe_ann": sharpe, "n": len(x)}


val_ls = {n: long_short(val, pred_val[n]) for n in pred_val}
best = max(val_ls, key=lambda n: summarise(val_ls[n], n)["sharpe_ann"])
print("best on val Sharpe:", best)

board = [summarise(long_short(test, pred_test[n]), f"EW LS {n}") for n in pred_test]
print(pd.DataFrame(board).round(4).to_string(index=False))

## 5. Long leg vs short leg (the L-shape)

In equities the short leg often carries the anomaly. The paper says crypto is the other way around: the **top** predicted quintile earns, the rest is mediocre.

Classroom check on the Combo forecast, test window: mean return of Q5, of Q1, and of the long–short.

In [ ]:
tmp = test[["date", "ret"]].copy()
tmp["yhat"] = pred_test["Combo"]
legs = []
for dt, g in tmp.groupby("date"):
    q1, q5 = g["yhat"].quantile([0.2, 0.8])
    legs.append({
        "date": dt,
        "long": g.loc[g["yhat"] >= q5, "ret"].mean(),
        "short": g.loc[g["yhat"] <= q1, "ret"].mean(),
        "mid": g.loc[(g["yhat"] > q1) & (g["yhat"] < q5), "ret"].mean(),
    })
legs = pd.DataFrame(legs).set_index("date")
print(pd.DataFrame([
    summarise(legs["long"], "Combo Q5 (long)"),
    summarise(-legs["short"], "Combo −Q1 (short contribution)"),
    summarise(legs["long"] - legs["short"], "Combo Q5−Q1"),
    summarise(legs["mid"], "Combo Q2–Q4"),
]).round(4).to_string(index=False))

## 6. Value-weight, costs, and dropping hard-to-trade names

Paper: combination VW Sharpe **1.66**; costs eat more than half the long–short but leave a remainder; alphas are **3–4×** larger in the hardest-to-trade tercile.

Classroom stand-in on Combo, test window.

In [ ]:
ls_ew = long_short(test, pred_test["Combo"])
ls_vw = long_short(test, pred_test["Combo"], weight="mcap")
ls_cost = long_short(test, pred_test["Combo"], cost=0.0020)
easy = test[test["hard"] < 0.5]
hard = test[test["hard"] > 0.5]
pred_easy = {k: v[test["hard"].to_numpy() < 0.5] for k, v in pred_test.items()}
pred_hard = {k: v[test["hard"].to_numpy() > 0.5] for k, v in pred_test.items()}

cmp = [
    summarise(ls_ew, "EW Combo"),
    summarise(ls_vw, "VW Combo"),
    summarise(ls_cost, "EW Combo minus 20bp/leg"),
    summarise(long_short(easy, pred_easy["Combo"]), "EW Combo, easy names only"),
    summarise(long_short(hard, pred_hard["Combo"]), "EW Combo, hard-to-trade only"),
]
print(pd.DataFrame(cmp).round(4).to_string(index=False))

cum = pd.DataFrame({
    "EW": (1 + ls_ew).cumprod(),
    "VW": (1 + ls_vw).cumprod(),
    "EW after 20bp": (1 + ls_cost).cumprod(),
})
ax = cum.plot(title="OOS growth of $1 — Combo long–short")
ax.set_ylabel("Growth of $1")
plt.tight_layout()
plt.show()

## 7. Discussion questions

1. On $R^2_{\mathrm{OOS}}$ vs Sharpe: can a “disappointing” $R^2$ still sort winners from losers? That is the paper’s first numerical point.
2. Why can **OLS** beat RF / MLP here, and why is that the **opposite** of notebook `01a` / Gu?
3. Where does the Combo long–short come from — Q5, Q1, or both? Equities vs crypto.
4. Drop the hard-to-trade names. What is left? Map to Avramov et al.: liquidity and limits to arbitrage keep the pattern alive.
5. The paper’s combination VW Sharpe is **1.66** before a serious implementability discussion (70–80% drawdowns; tiny coins). What would you refuse to call a tradable alpha?
6. Notebook `05` classifies **delisting**. This notebook *forecasts* next-week return. Same market — two economic questions.

## 8. Experiments — "do differently"

Uncomment one. Keep the validation window honest.

In [ ]:
# # Fit on the four signal columns only (drop the noise z*)
# # models_s = ...  FEATURES -> SIGNAL

# # Two-week hold: form the LS every other week (paper: ~30% less profit)
# ls_bi = ls_ew.iloc[::2]
# print(pd.DataFrame([summarise(ls_bi, "EW Combo, every other week")]).round(4).to_string(index=False))

# # Winsorize test returns at 1/99 before the LS (paper caveat: extreme volatile coins)
# tw = test.copy()
# lo, hi = tw["ret"].quantile([0.01, 0.99])
# tw["ret"] = tw["ret"].clip(lo, hi)
# print(pd.DataFrame([summarise(long_short(tw, pred_test["Combo"]), "EW Combo, winsorized ret")]).round(4).to_string(index=False))